# A01 — Code Interpreter: arquivos e estatísticas

![Arquitetura do exercício](assets/code-files.png)

## O que você vai aprender

Você abre uma sessão, envia o CSV e o script, lista os arquivos e executa Python na AWS. O Code Interpreter fornece o ambiente isolado e devolve stdout, stderr e exitCode. Este exemplo não usa um LLM.

## Pré-requisitos e sequência

Use uma conta AWS com sessão autorizada em `us-west-2`, Python 3.12 e kernel próprio para este apêndice.
As células instalam as versões validadas e criam os recursos durante a execução. Execute de cima para baixo, incluindo a limpeza.
Uma conta basta; as permissões variam conforme o evento. Estes exemplos são independentes das aulas Santander.

## Como interpretar o resultado

A resposta deve listar `data.csv` e `stats.py`, apresentar estatísticas das 299.130 linhas sintéticas e terminar com `exitCode=0`. A validação confere a resposta da ferramenta, e a última célula encerra a sessão.

## Origem e adaptações

Derivado do [notebook oficial](https://github.com/awslabs/agentcore-samples/blob/0d46da96306fa692c9887d0461252bf5aeeee6c3/06-workshops/05-AgentCore-tools/01-Agent-Core-code-interpreter/01-file-operations-using-code-interpreter/file-operations-using-code-interpreter.ipynb), revisão `0d46da96306f`.
Foram adicionados explicações, validações e dependências explícitas; os caminhos dos arquivos foram organizados neste apêndice.
O Browser usa a cadeia padrão de credenciais e nomes exclusivos. O exemplo de cálculo declara Claude Haiku 4.5.
As evidências e diferenças constam no [relatório de validação](VALIDACAO.md).

## Conferir a sessão AWS


In [ ]:
import boto3, os
session = boto3.Session()
REGION = session.region_name or os.environ.get("AWS_DEFAULT_REGION")
assert REGION == "us-west-2", "Este apêndice foi validado em us-west-2; configure a região antes de continuar."
identity = session.client("sts").get_caller_identity()
print({"conta": identity["Account"], "regiao": REGION, "identidade": identity["Arn"]})
# Credenciais vêm da cadeia padrão do boto3: role do ambiente, perfil ou sessão temporária.
# Nenhuma chave, senha ou token deve ser salvo neste notebook.


In [ ]:
%pip install "bedrock-agentcore==1.25.3" "boto3==1.43.124" "strands-agents==1.64.0"


In [ ]:
from bedrock_agentcore.tools.code_interpreter_client import CodeInterpreter
import json
from typing import Dict, Any

# Initialize the Code Interpreter with your AWS region
code_client = CodeInterpreter("us-west-2")
code_client.start()

## 2. Reading Local Files

Now we'll read the contents of our sample data file and analysis script.

In [ ]:
def read_file(file_path: str) -> str:
    """Helper function to read file content with error handling"""
    try:
        with open(file_path, "r", encoding="utf-8") as file:
            return file.read()
    except FileNotFoundError:
        print(f"Error: The file '{file_path}' was not found.")
        return ""
    except Exception as e:
        print(f"An error occurred: {e}")
        return ""


# Read both files
data_file_content = read_file("assets/data.csv")
code_file_content = read_file("assets/stats.py")

## 3. Preparing Files for Sandbox Environment

We'll create a structure that defines the files we want to create in the sandbox environment.

In [ ]:
files_to_create = [
    {"path": "data.csv", "text": data_file_content},
    {"path": "stats.py", "text": code_file_content},
]

## 4. Creating Helper Function for Tool Invocation

This helper function will make it easier to call sandbox tools and handle their responses. Within an active session, you can execute code in supported languages (Python, JavaScript), access libraries based on your dependencies configuration, generate visualizations, and maintain state between executions.

In [ ]:
def call_tool(tool_name: str, arguments: Dict[str, Any]) -> Dict[str, Any]:
    """Helper function to invoke sandbox tools

    Args:
        tool_name (str): Name of the tool to invoke
        arguments (Dict[str, Any]): Arguments to pass to the tool

    Returns:
        Dict[str, Any]: JSON formatted result
    """
    response = code_client.invoke(tool_name, arguments)
    for event in response["stream"]:
        return json.dumps(event["result"])

## 5. Writing Files to Sandbox

Now we'll write our files into the sandbox environment and verify they were created successfully.

In [ ]:
# Write files to sandbox
writing_files = call_tool("writeFiles", {"content": files_to_create})
print("Writing files result:")
print(writing_files)

# Verify files were created
listing_files = call_tool("listFiles", {"path": ""})
print("\nFiles in sandbox:")
print(listing_files)

## 6. Executing Analysis

Now we'll execute our analysis script in the sandbox environment and process the results.

In [ ]:
import pprint

# Execute the analysis script
code_execute_result = call_tool(
    "executeCode",
    {"code": files_to_create[1]["text"], "language": "python", "clearContext": True},
)

# Parse and display results
analysis_results = json.loads(code_execute_result)
print("Full analysis results:")
pprint.pprint(analysis_results)

print("\nStandard output from analysis:")
print(analysis_results["structuredContent"]["stdout"])

## 7. Cleanup

Finally, we'll clean up by stopping the Code Interpreter session. Once finished using a session, the session should be shopped to release resources and avoid unnecessary charges.

## Validar o resultado observado

As verificações abaixo falham se o resultado real não corresponder ao exercício.

In [ ]:
assert data_file_content.strip() and code_file_content.strip(), "Missing input fixtures"
for label, raw in (("write", writing_files), ("list", listing_files), ("execute", code_execute_result)):
    result = json.loads(raw)
    assert result.get("isError") is False, f"{label} failed: {result}"
assert "data.csv" in listing_files and "stats.py" in listing_files
assert analysis_results["structuredContent"]["exitCode"] == 0
assert "Traceback" not in analysis_results["structuredContent"]["stdout"]
assert analysis_results["structuredContent"]["stdout"].strip()
print("APPENDIX_RESULT_VALIDATED: AWS files created, listed and statistics script executed successfully")

In [ ]:
# Stop the Code Interpreter session
code_client.stop()
print("Code Interpreter session stopped successfully!")

## Conclusion

In this tutorial, we've learned how to:
- Initialize a Code Interpreter session
- Read and prepare files for analysis
- Execute code in a sandbox environment
- Process and display results
- Clean up resources